# 01 – Business Understanding

**Project:** Diabetes Prediction ML
**Seminar:** Advanced Applied Data Science – Goethe Universität Frankfurt
**Dataset:** CDC BRFSS 2015 – Diabetes Health Indicators (UCI #891)
**Target:** `Diabetes_binary` (0 = No Diabetes, 1 = Prediabetes/Diabetes)
**CRISP-DM Phase:** 1 – Business Understanding

---

## Ziel dieses Notebooks

Vor jeder Datenexploration oder Modellierung klären wir, **was** wir eigentlich vorhersagen wollen, **für wen**, **wie** das Ergebnis genutzt wird und **welche Metriken** dafür sinnvoll sind. Alle weiteren Notebooks (EDA, Preprocessing, Modeling, Evaluation) bauen auf diesen Entscheidungen auf.

## 1. Use Case

**Risikostratifizierung / Bevölkerungs-Screening.**

Eine Person beantwortet im Web ein paar Fragen zu Gesundheitszustand und Lebensstil (BMI, Alter, bekannter Bluthochdruck, Bewegung, allgemeines Wohlbefinden, …). Das Tool gibt eine **Risiko-Einschätzung** für Prediabetes/Diabetes zurück und eine gestaffelte Handlungsempfehlung. Das Modell ersetzt keine ärztliche Diagnose, sondern empfiehlt bei erhöhtem Risikoprofil die ärztliche Abklärung.

**Warum dieser Use Case zum Datensatz passt:**
- BRFSS ist selbst eine selbstberichtete Telefon-Umfrage → Trainings- und Anwendungsverteilung sind identisch.
- Keine klinischen Biomarker nötig (kein HbA1c, keine Glukose).
- Niedrigschwellige Empfehlung („zum Arzt für Bluttest" oder „Lifestyle anpassen") ist kostenarm und risikoarm.

**Einordnung von Stroke und HeartDiseaseorAttack:** Beide Variablen sind im BRFSS als selbstberichtete Diagnosen erfasst — sie sind beobachtbare Komorbiditätsindikatoren, keine kausalen Frühpräventionsvariablen. Ihr Vorhandensein im Feature-Set verschiebt die Interpretation des Modells von reiner Früherkennung hin zur allgemeinen Risikobewertung bestehender Hochrisikoprofile. Das ist im BRFSS-Screening-Kontext methodisch zulässig.

## 2. Stakeholder

- **Primär:** Endnutzer (Privatpersonen, die ihr Risiko einschätzen wollen).
- **Sekundär:** Public-Health-Kontext (Bevölkerungs-Screening, Aufklärung).

## 3. Cost Matrix

| Vorhersage \ Wahrheit | Diabetes/Prediabetes | Kein Diabetes |
|---|---|---|
| **Modell: erhöhtes Risiko** | TP: Person sucht ärztliche Abklärung → ggf. Diagnose und Frühintervention. **Großer Nutzen.** | FP: Arztbesuch ohne Befund. **Geringe Kosten.** Bei Hochrisiko-Profil sogar nützliche Vorsorge. |
| **Modell: niedriges Risiko** | FN: Person unerkannt. **Hohe Kosten** (Spätfolgen, verpasste Frühintervention). | TN: korrekt entwarnt. |

**Konsequenz:** FN ≫ FP → **Recall-Priorität.** Wir akzeptieren niedrigere Precision zugunsten von hohem Recall.

## 4. Label-Noise: zentrale methodische Einsicht

### 4.1 Wie das Target im Datensatz erfasst wird

Die Zielvariable basiert auf der BRFSS-2015-Frage „**(Ever told) you have diabetes**" mit dem Prolog „**Has a doctor, nurse, or other health professional ever told you that you had any of the following?**" — also reine **Selbstauskunft über eine ärztliche Diagnose**, nicht das Vorliegen von Diabetes selbst. Frauen, die nur während einer Schwangerschaft betroffen waren, sowie Personen mit „weiß nicht"/„keine Angabe" wurden in der UCI-Variante zu den Nicht-Diabetikern gezählt; „Yes" und „pre-diabetes/borderline" wurden zu `Diabetes_binary = 1` zusammengefasst.

Quelle: [BRFSS 2015 Codebook, Variable `DIABETE3`](https://www.cdc.gov/brfss/annual_data/2015/pdf/codebook15_llcp.pdf).

→ Das Label kodiert also **Diagnose-Status**, nicht zwingend **Krankheits-Status**. Negative Labels können undiagnostizierte Fälle enthalten. Das ist eine plausible methodische Einschränkung auf Basis bekannter Unterdiagnose-Raten — keine direkt im Datensatz beobachtete Tatsache.

### 4.2 Anteil undiagnostizierter Fälle

Ein erheblicher Anteil der US-Bevölkerung weiß nicht, dass er von Diabetes oder Prediabetes betroffen ist:

- **Diabetes:** ~27,6 % der Erwachsenen mit Diabetes in den USA sind undiagnostiziert (entspricht ~11 Millionen Personen, 2023). Quelle: [CDC National Diabetes Statistics Report](https://www.cdc.gov/diabetes/php/data-research/index.html), Januar 2026.
- **Prediabetes:** ~80 % der Erwachsenen mit Prediabetes wissen nicht, dass sie es haben. Quelle: [CDC – A U.S. Report Card on Diabetes](https://www.cdc.gov/diabetes/communication-resources/diabetes-statistics.html).
- **Validierung über unabhängige Datenquelle:** Im NHANES 2021–2023 ist die undiagnostizierte Diabetes-Prävalenz bei Erwachsenen ab 20 Jahren 4,2 % bei einer Gesamtprävalenz von 14,3 % → ~29 % der Diabetes-Population undiagnostiziert. Quelle: [NCHS Data Brief 516](https://www.cdc.gov/nchs/products/databriefs/db516.htm), November 2024.

### 4.3 Konsequenzen für unser Modell

- Viele „Negative" im Datensatz könnten real positiv sein (asymmetrisches Label-Noise auf der negativen Klasse — plausible Annahme, nicht direkt beobachtet).
- Das Modell wird möglicherweise für korrekte Vorhersagen bestraft, wenn es undiagnostizierte Hochrisiko-Profile erkennt.
- Die *gemessene* PR-AUC unterschätzt möglicherweise die *tatsächliche* Modellgüte.

**Im Bericht zu zeigen:** False-Positive-Profilanalyse — sehen unsere FPs aus wie eine undiagnostizierte Hochrisiko-Population (hoher BMI, HighBP, höheres Alter, schlechte GenHlth)? Ein Teil der FPs könnte auf undiagnostizierte oder im Label nicht erfasste Hochrisikofälle zurückgehen — das wäre ein Hinweis auf systematischen Label-Noise, nicht auf Modellfehler.

## 5. Metriken — Auswahl und Begründung

Dieser Abschnitt erklärt im Detail, **welche Metriken wir nutzen und warum**. Die Wahl ist nicht selbstverständlich: bei einem stark unbalancierten Datensatz wie diesem (~14 % positive Klasse) liefern die gängigen Standardmetriken irreführende Ergebnisse. Wir leiten unsere Entscheidung Schritt für Schritt her.

### 5.1 Fundament: Confusion Matrix

Alle Klassifikationsmetriken bauen auf vier Zahlen auf. Für eine binäre Klassifikation:

|   | Modell: positiv | Modell: negativ |
|---|---|---|
| **Wirklich positiv** | **TP** – richtig erkannt | **FN** – verpasst |
| **Wirklich negativ** | **FP** – falscher Alarm | **TN** – richtig entwarnt |

In medizinischem Screening ist **FN besonders teuer**: jemand mit Diabetes wird nach Hause geschickt, kein Follow-up. **FP ist meist billiger**: jemand bekommt eine unnötige zweite Untersuchung, aber kein bleibender Schaden. → siehe Cost Matrix oben.

### 5.2 Die einzelnen Metriken — jede beantwortet eine konkrete Frage

**Accuracy = (TP + TN) / Gesamt**
- Frage: *„Wie oft hat das Modell insgesamt recht?"*
- **Problem bei Imbalance:** Ein Modell, das immer „kein Diabetes" sagt, erreicht bei uns ~86 % Accuracy und ist klinisch wertlos. Alle Diabetiker werden übersehen.
- **→ Accuracy wird *nicht* als Hauptmetrik verwendet.**

**Precision = TP / (TP + FP)**
- Frage: *„Wenn das Modell Alarm schlägt, wie oft hat es recht?"*
- Hoch = wenige falsche Alarme.
- Hängt vom gewählten Schwellenwert ab.

**Recall (Sensitivität) = TP / (TP + FN)**
- Frage: *„Wie viele der echten Diabetiker erwische ich?"*
- Hoch = wenige verpasste Fälle.
- **In medizinischem Screening fast immer die wichtigste Größe.**
- Hängt vom gewählten Schwellenwert ab.

**Specificity = TN / (TN + FP)**
- Frage: *„Wie viele der Gesunden lasse ich korrekt in Ruhe?"*
- Das Gegenstück zu Recall, fokussiert auf die negative Klasse.

**F1 = 2 · (Precision · Recall) / (Precision + Recall)**
- Harmonisches Mittel von Precision und Recall.
- Wird klein, sobald eine der beiden Größen klein ist.
- **Nachteil:** hängt vom gewählten Schwellenwert ab → sagt wenig über das Modell „als Ganzes".

### 5.3 Wichtige Einsicht: Precision und Recall hängen am Schwellenwert

Ein Klassifikationsmodell gibt eigentlich keine 0/1-Vorhersage, sondern eine **Wahrscheinlichkeit zwischen 0 und 1**. Wir selbst legen fest, ab welcher Schwelle wir „positiv" sagen.

- Schwelle = 0,5 (Default): mittlere Precision, mittlerer Recall.
- Schwelle = 0,2: mehr Personen werden als positiv markiert → **Recall steigt, Precision sinkt**.
- Schwelle = 0,8: nur sehr sichere Fälle werden markiert → **Precision steigt, Recall sinkt**.

Es gibt also nicht *eine* Precision und *einen* Recall — es gibt eine **ganze Kurve**. Eine sinnvolle Hauptmetrik sollte daher **schwellenunabhängig** sein und die Modell-Qualität über alle möglichen Schwellen bewerten.

### 5.4 ROC-AUC vs. PR-AUC — was die Kurven plotten

Beide Kurven entstehen, indem der Schwellenwert von 1,0 langsam auf 0,0 gesenkt wird und bei jedem Schritt zwei Zahlen berechnet werden:

- **ROC-Kurve:** Recall (= TPR) gegen False-Positive-Rate (FPR = FP / (FP + TN))
- **PR-Kurve:** Precision gegen Recall

Die Fläche unter der jeweiligen Kurve ist die „AUC". Der entscheidende Unterschied liegt darin, **welche Zahlen im Nenner stehen**:

- Recall = TP / (TP + FN) → Nenner = alle echten Positiven
- FPR = FP / (FP + TN) → **Nenner = alle echten Negativen** (riesig bei Imbalance!)
- Precision = TP / (TP + FP) → Nenner = alle Modell-Positiven (klein)

### 5.5 Konkretes Beispiel — warum ROC-AUC bei Imbalance weniger aussagekräftig ist

Stellen wir uns 1.000 Personen vor: 140 echte Diabetiker, 860 ohne Diabetes (entspricht unserer Klassenverteilung).

Das Modell markiert 200 Personen als positiv. Davon sind 100 wirklich Diabetiker (TP) und 100 falscher Alarm (FP):
- TP = 100, FN = 40, FP = 100, TN = 760

Berechnete Metriken:

| Metrik | Wert | Interpretation |
|---|---|---|
| **Recall** | 100 / 140 = **0,71** | 71 % der Diabetiker erwischt |
| **Precision** | 100 / 200 = **0,50** | jeder zweite Alarm ist falsch |
| **FPR** | 100 / 860 = **0,12** | *sieht klein aus* |

**Die FPR beträgt nur 0,12 — obwohl die Hälfte aller Alarme falsch ist.**

Warum? Weil im Nenner 860 stehen. 100 Fehler auf 860 Negative wirken gering. → **ROC-AUC ist bei Imbalance weniger sensitiv gegenüber Verbesserungen auf der positiven Klasse**, weil die große negative Klasse den FPR-Nenner dominiert.

Precision dagegen zeigt das Problem direkt: 0,50 sagt klar „die Hälfte ist falsch". → **PR-AUC nutzt keine TN-Zahl im Nenner und bleibt dadurch fokussiert auf die Minderheitsklasse.**

**Vergleich bei ausgeglichenen Daten** (500/500, gleiche Vorhersagen):
- FPR = 100 / 500 = **0,20** statt 0,12 → das Problem würde sich in der ROC stärker zeigen.
- Precision unverändert 0,50.

**→ ROC-AUC ist ein valides Maß, wird aber bei starker Imbalance durch die große negative Klasse abgemildert. PR-AUC ist auch bei Imbalance sensitiv für Verbesserungen auf der positiven Klasse.**

### 5.6 Baselines im Vergleich

| Metrik | Zufalls-Baseline | Anmerkung |
|---|---|---|
| **ROC-AUC** | immer 0,5 | unabhängig von der Klassenverteilung |
| **PR-AUC** | = Klassenprävalenz | bei uns **≈ 0,14** |

Konsequenz für die Einordnung:
- Wenn jemand „ROC-AUC = 0,80" reportet, sagt das bei 14 % Prävalenz nur, dass das Modell *irgendwie* trennt. Es sagt wenig darüber, ob bei einer praktisch nutzbaren Schwelle die Precision brauchbar ist.
- **PR-AUC = 0,44 vs. Baseline 0,14** sagt direkt: ~3× besser als Zufall, in genau der Disziplin, die hier zählt.

### 5.7 Resultierende Metrik-Strategie

| Metrik | Rolle | Begründung |
|---|---|---|
| **PR-AUC (Average Precision)** | **Primär** (Modellauswahl) | Schwellenunabhängig, fokussiert auf Minderheitsklasse, robust gegen Imbalance. No-Skill-Baseline = 0,14. |
| ROC-AUC | Sekundär (Vergleich) | Standard in der Literatur, ermöglicht Einordnung. Bei Imbalance leicht optimistisch verzerrt — nicht als alleinige Entscheidungsgröße. |
| Confusion Matrix + Precision/Recall/F1 @ Threshold | **Pflicht** (binärer Output) | Standard-Reporting des binären Klassifikators an dem in Section 6 gewählten Threshold. |
| Subgruppen-Performance (PR-AUC pro Gruppe) | Sekundär (Fairness) | Tool muss über Geschlecht, Alter, Einkommen vergleichbar funktionieren. |
| Accuracy | **nicht** als Hauptmetrik | Irreführend bei Imbalance (86 % durch Trivial-Modell). Nur als Kontextzahl. |
| F1 | nur an gewählter Schwelle | Schwellenwert-abhängig, daher nicht zur Modellauswahl. |

### Workflow

1. **Modellauswahl** mit PR-AUC (Cross-Validation).
2. **Threshold setzen** anhand Use-Case-Constraint (siehe Section 6).
3. **Standard-Binär-Evaluation** an diesem Threshold (Confusion Matrix, Precision, Recall, F1).
4. **Subgruppen-Auswertung** über demographische Variablen.

## 6. Klassifikations-Strategie

Das Modell liefert für jede Person eine Wahrscheinlichkeit `p ∈ [0, 1]`. Für die binäre Klassifikation wird **ein** Threshold gesetzt:

`ŷ = 1` falls `p ≥ T`, sonst `ŷ = 0`.

### Wahl des Thresholds

Da unser Use Case ein Screening-Tool ist, bei dem False Negatives deutlich teurer sind als False Positives (siehe Cost Matrix in Section 3), wählen wir den Threshold nicht naiv bei 0,5, sondern **Use-Case-getrieben**:

> *T = der kleinste Wert, bei dem auf den CV-Validierungsfolds ein Recall ≥ 0,80 erreicht wird.*

Begründung: Bei einem Screening-Tool ist es wichtiger, möglichst viele wirklich Betroffene zu identifizieren (hoher Recall) als jeden Alarm präzise zu rechtfertigen. Die konkrete Recall-Zielmarke (0,7 / 0,8 / 0,9) ist Team-Entscheidung mit Begründung.

**Wichtig:** Der Threshold wird ausschließlich auf den CV-Validierungsfolds gewählt — niemals auf dem Testset. Das Testset wird einmalig und final für die Evaluation des fertig trainierten Modells genutzt. Jede Nutzung des Testsets zur Threshold-Wahl, Modellauswahl oder Feature-Selektion würde Data Leakage erzeugen und die berichtete Testperformance verzerren.

### Output am Threshold (Pflicht)

- Confusion Matrix
- Accuracy (mit Hinweis auf Imbalance-Bias, als Kontextzahl)
- Precision, Recall, F1 (positive Klasse)
- Precision, Recall, F1 (negative Klasse)
- Classification Report (sklearn-Standard)
- Plus schwellenunabhängig: PR-AUC, ROC-AUC

Damit ist der binäre Klassifikator vollständig dokumentiert und entspricht der Standard-Erwartung an ein Supervised-Learning-Projekt.

## 7. Erfolgsdefinition

Das Modell ist als binärer Klassifikator einsatzfähig, wenn:

1. **PR-AUC signifikant über No-Skill-Baseline** (0,14). Die genaue erreichbare Höhe wird empirisch ermittelt, nicht vorab fixiert.
2. **Am gewählten Threshold:** Recall erreicht das im Recall-Constraint definierte Ziel (z. B. ≥ 0,80); Precision wird transparent reportet, nicht als Zielgröße vorgegeben.
3. **Subgruppen-Performance vergleichbar** über Geschlecht, Altersgruppen, Einkommen (kein dramatischer Bias).
4. **Modell interpretierbar** (Feature Importance / SHAP — welche Faktoren treiben die Vorhersage?).

## 8. Limitationen

- **Selbstauskunft:** Antworten können falsch, unvollständig oder verzerrt sein.
- **Label-Noise:** „Negative" enthalten undiagnostizierte Fälle → Performance-Obergrenze.
- **Datenalter:** BRFSS 2015 → Population, Lebensstil und Risikoverteilungen können sich verschoben haben.
- **Geografie:** USA + Puerto Rico → Übertragbarkeit auf andere Länder eingeschränkt.
- **Survey-Bias:** Telefonumfrage erreicht bestimmte Gruppen schlechter (Jüngere, Niedrigeinkommen).
- **Coarse Features:** Keine echten Biomarker → harte methodische Decke.
- **Kein klinisches Diagnosetool:** Liefert nur eine Empfehlung zur Abklärung, ersetzt keine ärztliche Diagnose.

## 9. Methodisches Alleinstellungsmerkmal

- **Use-Case-getriebene Threshold-Wahl** (Recall-Constraint) statt Default 0,5 — begründet aus Cost Matrix.
- **Label-Noise-Diskussion**: BRFSS-Label kodiert Diagnose-Status, nicht Krankheits-Status; gemessene Performance ist deshalb eine Untergrenze für die wahre Modellgüte.
- **Saubere Trennung** von schwellenunabhängiger Modellauswahl (PR-AUC) und schwellenabhängiger binärer Klassifikation.
- **Fairness-/Subgruppen-Analyse** auf einem realen Survey-Datensatz mit demographischen Variablen.
- **Bewusste Metrik-Wahl**: Accuracy verworfen, PR-AUC primär, ROC-AUC nur sekundär — jede Entscheidung an der Klassenverteilung begründet.

## 10. Optionale Erweiterungen

Die folgenden Punkte sind **keine Pflichtbestandteile** des binären Klassifikations-Projekts. Sie können das Projekt methodisch vertiefen, falls Zeit und Scope es zulassen. Jeder Punkt ist eigenständig und kann unabhängig behandelt oder weggelassen werden.

### 10.1 Kalibrierungsanalyse (Brier-Score, Calibration Plot)

PR-AUC und ROC-AUC messen **Diskriminierung** — wie gut das Modell positive von negativen Fällen trennt. Sie sagen aber nichts darüber, ob die ausgegebenen Wahrscheinlichkeiten **realistisch** sind. Eine Vorhersage „p = 0,70" sollte tatsächlich bedeuten, dass von 100 Personen mit diesem Score etwa 70 wirklich Diabetes haben.

**Methodik:**
- Calibration Plot (Reliability Diagram): vorhergesagte Wahrscheinlichkeit gegen tatsächlichen Anteil positiver Fälle; perfekt = Diagonale.
- Brier-Score: mittlerer quadratischer Fehler zwischen Vorhersage und 0/1-Label.
- Bei Bedarf Nachkalibrierung mit Platt Scaling oder Isotonic Regression (besonders für Random Forest / XGBoost / SVM).

**Warum optional:** Für die reine binäre Klassifikation an einem festen Threshold ist Kalibrierung nicht zwingend. Sie wird relevant, sobald die kontinuierliche Wahrscheinlichkeit selbst kommuniziert oder für gestaffelte Entscheidungen genutzt wird (siehe 10.2).

### 10.2 Tier-basierte Deployment-Erweiterung

Statt einer binären 0/1-Ausgabe könnte das Tool eine **gestaffelte Empfehlung** in drei Stufen geben, basierend auf zwei Thresholds:

| Risiko-Score (p) | Tier | Empfehlung |
|---|---|---|
| `p < T_low` | Niedrig | Lifestyle beibehalten, Standard-Check-ups. |
| `T_low ≤ p < T_high` | Erhöht | Lifestyle-Anpassung, bei nächstem Arztbesuch ansprechen. |
| `p ≥ T_high` | Hoch | Zeitnaher HbA1c-Test beim Hausarzt. |

**Wahl von `T_high`:** balancierter Trade-off, z. B. Recall ≥ 0,70 bei Precision ≥ 0,35 (Team-Entscheidung).

**Konsistenz:** Identisches Modell, identisches binäres Trainings-Target — nur zusätzliche Schwellen-Interpretation des kontinuierlichen Outputs. Setzt sinnvolle Kalibrierung (10.1) voraus.

**Warum optional:** Erweitert den Pflichtteil (binäre Klassifikation) um eine deployment-orientierte Sicht, ist aber kein Ersatz für ihn.

### 10.3 False-Positive-Profilanalyse (Label-Noise-Vertiefung)

Aus der Label-Noise-Einsicht in Section 4 folgt eine testbare Hypothese: Viele False Positives unseres Modells könnten in Wahrheit **undiagnostizierte Risiko-Profile** sein, nicht Modellfehler.

**Methodik:**
- FP-Subgruppe extrahieren und deren Feature-Verteilung mit der TP-Subgruppe vergleichen.
- Erwartung bei valider Hypothese: FPs ähneln TPs in den klassischen Risikofaktoren (BMI, HighBP, höheres Alter, GenHlth) → Hinweis auf undiagnostizierten Diabetes-Status.
- Ergebnis stützt die Argumentation, dass gemessene Performance eine **Untergrenze** der wahren Modellgüte ist.

**Warum optional:** Vertieft Section 4 empirisch, ist aber für die Pflicht-Evaluation nicht erforderlich.

## Zusammenfassung

Wir entwickeln einen binären Klassifikator zur Diabetes-Risikoabschätzung auf Basis selbstberichteter BRFSS-Lifestyle- und Gesundheitsdaten. Das Target ist `Diabetes_binary` (0/1); die Klassen sind stark unbalanciert (~14 % positiv). Als Primärmetrik nutzen wir **PR-AUC**, weil sie bei Imbalance robust und auf die Minderheitsklasse fokussiert ist; ROC-AUC dient nur als Sekundärmetrik zur Literaturvergleichbarkeit. Der Klassifikations-Threshold wird **use-case-getrieben** über einen Recall-Constraint gewählt (Screening-Logik, FN ≫ FP), nicht naiv bei 0,5. Standard-Output ist die binäre Klassifikation mit vollständigem Reporting (Confusion Matrix, Precision/Recall/F1, PR-AUC, ROC-AUC). Methodisch zentral ist die Einsicht, dass das BRFSS-Label **Diagnose-Status** kodiert, nicht **Krankheits-Status** — daraus folgt asymmetrisches Label-Noise auf der negativen Klasse und eine Performance-Untergrenze, die im Bericht ehrlich kontextualisiert wird. Mögliche methodische Vertiefungen (Kalibrierung, Tier-Deployment, FP-Profilanalyse) sind in Section 10 als optionale Erweiterungen dokumentiert.

---

**Nächster Schritt:** `02_data_understanding.ipynb` — echte EDA mit Duplikat-/Label-Konsistenz-Analyse, Outlier-Inspektion, Feature-Target-Beziehungen und Class-Overlap.